# 🌐 UTe₂ Hybrid AI-Powered Simulation Dashboard

Interactive dashboard for hybrid superconductivity simulations with offline-first execution.

## Features:
- 🔧 Interactive parameter controls
- 🎯 Multiple execution modes (Offline/Local/Hybrid/Cloud)
- 📊 Real-time visualization with Plotly
- 🤖 AI-powered optimization suggestions
- 💾 Persistent caching for faster reruns
- 🌐 Connection-aware mode switching

In [ ]:
# Import required libraries
import yaml
import json
import logging
import numpy as np
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Import UTe2 system modules
from ute2_hybrid_engine import HybridExecutionEngine, ExecutionMode
from ute2_simulation import UTe2SimulationEngine
from ute2_ai_advisor import AIAdvisor
from ute2_cache import SimulationCache

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Load configuration
config_path = Path('hybrid_config.yaml')
if not config_path.exists():
    print("⚠️ Configuration file not found! Using default settings.")
    config = {}
else:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print("✅ Configuration loaded successfully")

# Initialize hybrid execution engine
engine = HybridExecutionEngine(config)
print(f"✅ Hybrid Execution Engine initialized (mode: {engine.current_mode.value})")

## 🎛️ Interactive Control Panel

In [ ]:
# Create interactive widgets
style = {'description_width': '150px'}
layout = widgets.Layout(width='500px')

# Temperature slider
temp_slider = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=3.0,
    step=0.1,
    description='Temperature (K):',
    style=style,
    layout=layout,
    continuous_update=False
)

# Magnetic field slider
field_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=80.0,
    step=1.0,
    description='Magnetic Field (T):',
    style=style,
    layout=layout,
    continuous_update=False
)

# K-grid size slider
kgrid_slider = widgets.IntSlider(
    value=32,
    min=8,
    max=128,
    step=8,
    description='K-grid Size:',
    style=style,
    layout=layout,
    continuous_update=False
)

# Execution mode dropdown
mode_dropdown = widgets.Dropdown(
    options=['Offline', 'Local', 'Hybrid', 'Cloud'],
    value='Local',
    description='Execution Mode:',
    style=style,
    layout=layout
)

# Run button
run_button = widgets.Button(
    description='🚀 Run Simulation',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

# Status indicator
status_html = widgets.HTML(value="<b>Status:</b> Ready")

# Online/Offline indicator
connection_html = widgets.HTML(
    value="<span style='color: orange;'>🔴 Offline Mode</span>"
)

# Output area for results
output_area = widgets.Output()

# AI Suggestions area
suggestions_output = widgets.Output()

print("✅ Interactive widgets created")

In [ ]:
# Define simulation execution function
def run_simulation(b):
    """Run simulation with current parameters"""
    with output_area:
        clear_output(wait=True)
        
        # Update status
        status_html.value = "<b>Status:</b> <span style='color: orange;'>Running...</span>"
        
        # Get parameters from widgets
        params = {
            'temperature_K': temp_slider.value,
            'magnetic_field_T': field_slider.value,
            'k_grid_size': kgrid_slider.value
        }
        
        # Set execution mode
        engine.set_mode(mode_dropdown.value)
        
        print(f"\n{'='*60}")
        print(f"Running UTe₂ Simulation")
        print(f"{'='*60}")
        print(f"Temperature: {params['temperature_K']:.2f} K")
        print(f"Magnetic Field: {params['magnetic_field_T']:.1f} T")
        print(f"K-grid Size: {params['k_grid_size']}")
        print(f"Mode: {mode_dropdown.value}")
        print(f"{'='*60}\n")
        
        # Submit and execute task
        task_id = engine.submit_task(params, priority=1)
        result = engine.execute_next_task()
        
        if result and result.success:
            status_html.value = "<b>Status:</b> <span style='color: green;'>✅ Success</span>"
            
            print(f"✅ Simulation completed in {result.execution_time:.2f}s")
            print(f"Cache hit: {result.metadata.get('cache_hit', False)}")
            
            # Visualize results
            visualize_results(result)
        else:
            status_html.value = "<b>Status:</b> <span style='color: red;'>❌ Failed</span>"
            print("❌ Simulation failed")
            if result:
                print(f"Error: {result.metadata.get('error', 'Unknown error')}")

def visualize_results(task_result):
    """Visualize simulation results using Plotly"""
    result = task_result.result
    
    if result.gap_function is None:
        print("No gap function data to visualize")
        return
    
    # Create subplots
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Superconducting Gap Distribution', 'Gap Statistics'),
        specs=[[{'type': 'scatter'}, {'type': 'bar'}]]
    )
    
    # Gap function plot
    gap_values = result.gap_function
    n_points = min(len(gap_values), 1000)  # Limit for performance
    indices = np.linspace(0, len(gap_values)-1, n_points, dtype=int)
    
    fig.add_trace(
        go.Scatter(
            y=gap_values[indices],
            mode='markers',
            marker=dict(size=3, color=gap_values[indices], colorscale='Viridis'),
            name='Gap'
        ),
        row=1, col=1
    )
    
    # Statistics
    stats = {
        'Mean': np.mean(gap_values),
        'Max': np.max(gap_values),
        'Min': np.min(gap_values),
        'Std': np.std(gap_values)
    }
    
    fig.add_trace(
        go.Bar(
            x=list(stats.keys()),
            y=list(stats.values()),
            marker=dict(color=['blue', 'green', 'red', 'orange'])
        ),
        row=1, col=2
    )
    
    fig.update_xaxes(title_text="K-point Index", row=1, col=1)
    fig.update_yaxes(title_text="Gap (eV)", row=1, col=1)
    fig.update_yaxes(title_text="Value (eV)", row=1, col=2)
    
    fig.update_layout(
        height=400,
        showlegend=False,
        template='plotly_dark'
    )
    
    fig.show()
    
    # Print summary
    print(f"\n📊 Gap Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value:.6f} eV")

# Connect button to function
run_button.on_click(run_simulation)

print("✅ Simulation function configured")

In [ ]:
# AI Suggestions function
def show_ai_suggestions(b):
    """Display AI-generated suggestions"""
    with suggestions_output:
        clear_output(wait=True)
        
        temp = temp_slider.value
        field = field_slider.value
        
        suggestions = engine.advisor.suggest_parameters(
            temperature=temp,
            magnetic_field=field,
            device_type='laptop'
        )
        
        print("🤖 AI Advisor Suggestions:")
        print("=" * 40)
        
        if suggestions:
            for key, value in suggestions.items():
                print(f"  {key}: {value}")
        else:
            print("  No suggestions available")
        
        # Get device recommendations
        device_recs = engine.advisor.get_device_recommendations('laptop')
        if device_recs and 'tips' in device_recs:
            print("\n💡 Tips:")
            for tip in device_recs['tips']:
                print(f"  • {tip}")

# AI suggestions button
ai_button = widgets.Button(
    description='🤖 Get AI Suggestions',
    button_style='info',
    layout=widgets.Layout(width='200px', height='40px')
)
ai_button.on_click(show_ai_suggestions)

print("✅ AI suggestions configured")

In [ ]:
# System status function
def show_system_status(b):
    """Display system status"""
    status = engine.get_status()
    cache_stats = status['cache_stats']
    advisor_stats = status['advisor_stats']
    
    print("\n📊 System Status:")
    print("=" * 50)
    print(f"Current Mode: {status['current_mode']}")
    print(f"Internet Available: {status['internet_available']}")
    print(f"Queued Tasks: {status['queued_tasks']}")
    print(f"Completed Tasks: {status['completed_tasks']}")
    print("\n💾 Cache:")
    print(f"  Enabled: {cache_stats['enabled']}")
    print(f"  Entries: {cache_stats['total_entries']}")
    print(f"  Size: {cache_stats['total_size_mb']:.2f} MB / {cache_stats['max_size_mb']} MB")
    print(f"  Total Accesses: {cache_stats['total_accesses']}")
    print("\n🤖 AI Advisor:")
    print(f"  Enabled: {advisor_stats['enabled']}")
    print(f"  Offline Mode: {advisor_stats['offline_mode']}")
    print(f"  Simulations Recorded: {advisor_stats['total_simulations_recorded']}")

# Status button
status_button = widgets.Button(
    description='📊 System Status',
    button_style='warning',
    layout=widgets.Layout(width='200px', height='40px')
)
status_button.on_click(show_system_status)

print("✅ Status display configured")

## 🎨 Dashboard Layout

In [ ]:
# Create dashboard layout
dashboard = widgets.VBox([
    widgets.HTML(value="<h2>🌐 UTe₂ Hybrid Simulation Dashboard</h2>"),
    widgets.HBox([status_html, connection_html]),
    widgets.HTML(value="<hr>"),
    widgets.HTML(value="<h3>Simulation Parameters</h3>"),
    temp_slider,
    field_slider,
    kgrid_slider,
    mode_dropdown,
    widgets.HTML(value="<hr>"),
    widgets.HBox([run_button, ai_button, status_button]),
    widgets.HTML(value="<hr>"),
    suggestions_output,
    output_area
])

# Display dashboard
display(dashboard)

print("\n✅ Dashboard ready! Use the controls above to run simulations.")

## 📈 Phase Diagram Generator

In [ ]:
# Phase diagram generation
def generate_phase_diagram():
    """Generate and display T-H phase diagram"""
    print("Generating phase diagram...")
    
    phase_data = engine.simulation_engine.generate_phase_diagram(
        temperature_range=(0.1, 2.5),
        field_range=(0, 80),
        resolution=50
    )
    
    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=phase_data['phase'],
        x=phase_data['magnetic_field'],
        y=phase_data['temperature'],
        colorscale='Viridis',
        colorbar=dict(
            title='Phase',
            tickvals=[0, 1, 2],
            ticktext=['Normal', 'SC', 'Reentrant SC']
        )
    ))
    
    fig.update_layout(
        title='UTe₂ Temperature-Field Phase Diagram',
        xaxis_title='Magnetic Field (T)',
        yaxis_title='Temperature (K)',
        template='plotly_dark',
        height=500
    )
    
    fig.show()
    print("✅ Phase diagram generated")

# Phase diagram button
phase_button = widgets.Button(
    description='📈 Generate Phase Diagram',
    button_style='primary',
    layout=widgets.Layout(width='250px', height='40px')
)
phase_button.on_click(lambda b: generate_phase_diagram())

display(phase_button)

## 🔧 Advanced Controls

In [ ]:
# Cache management
def clear_cache(b):
    """Clear simulation cache"""
    engine.cache.clear()
    print("✅ Cache cleared")

def show_cache_stats(b):
    """Show detailed cache statistics"""
    stats = engine.cache.get_stats()
    print("\n💾 Cache Statistics:")
    print("=" * 40)
    for key, value in stats.items():
        print(f"  {key}: {value}")

# Cache control buttons
clear_cache_button = widgets.Button(
    description='🗑️ Clear Cache',
    button_style='danger',
    layout=widgets.Layout(width='150px')
)
clear_cache_button.on_click(clear_cache)

cache_stats_button = widgets.Button(
    description='💾 Cache Stats',
    button_style='info',
    layout=widgets.Layout(width='150px')
)
cache_stats_button.on_click(show_cache_stats)

display(widgets.HBox([clear_cache_button, cache_stats_button]))

## 📝 Notes

### Execution Modes:
- **Offline**: Optimized for embedded devices (e.g., Raspberry Pi) with minimal resources
- **Local**: Full local execution with standard resources
- **Hybrid**: Automatically chooses between local and cloud based on task size
- **Cloud**: Offload all computations to cloud (requires internet)

### Tips:
- Use the AI Advisor for optimization suggestions based on your parameters
- Results are automatically cached for faster reruns
- The reentrant superconducting region appears at high fields (40-65 T)
- Lower k-grid sizes run faster but with reduced accuracy

### System Requirements:
- Python 3.8+
- numpy, scipy, plotly, ipywidgets, pyyaml
- See `requirements.txt` for full dependencies